# PWTA patch model: lightweight training demo

Runs the memory- and speed-optimized patch model (`patch_light.py`) on a few SMD machines
and reports segment-level recall at the anomaly-ratio budget.

Changes from the original pipeline (model, scoring and metric unchanged):
1. **Batched evaluation**: 71x less peak memory, bit-identical results.
2. **GPU-resident training data**: about 2x faster.
3. **Early stopping**: shorter training, nearly lossless.

## 1. Setup
Download SMD from [OmniAnomaly](https://github.com/NetManAIOps/OmniAnomaly) and set `SMD_ROOT`.

In [ ]:
import os, sys, time, numpy as np
sys.path.append("..")
from patch_light import patch_point_error, seg_spans, seg_detectability, CFG_LOSSLESS, CFG_FAST

SMD_ROOT = os.path.expanduser("OmniAnomaly/ServerMachineDataset")   # set to your local path

## 2. Data loading and per-machine recall

In [ ]:
def load_smd(m):
    tr = np.loadtxt(f"{SMD_ROOT}/train/{m}.txt", delimiter=",").astype(np.float32)
    te = np.loadtxt(f"{SMD_ROOT}/test/{m}.txt", delimiter=",").astype(np.float32)
    lb = np.loadtxt(f"{SMD_ROOT}/test_label/{m}.txt", delimiter=",").astype(int)
    return tr, te, lb

def machine_recall(m, cfg=CFG_LOSSLESS):
    tr, te, lb = load_smd(m)
    ps, cov = patch_point_error(tr, te, seed=0, normalize=False, cfg=cfg)
    lb_c = lb[:cov]; ar = float(lb_c.mean()); sp = seg_spans(lb_c)
    return float(np.mean([seg_detectability(ps, s, ar) for s in sp])) if sp else None

## 3. Demo: three SMD machines

In [ ]:
for m in ["machine-1-1", "machine-1-6", "machine-2-2"]:
    t0 = time.time(); r = machine_recall(m, CFG_LOSSLESS)
    print(f"{m}: recall_budget={r:.3f}  ({time.time()-t0:.1f}s, CFG_LOSSLESS)")

## 4. Configurations
- **`CFG_LOSSLESS`** (batch 128, GPU-resident, early stop): for final results; recall difference ~0.006 from the original settings.
- **`CFG_FAST`** (batch 256, 2x LR): for exploration; difference ~0.021.
- To keep peak memory below 1 GB, set `gpu_resident=False` for very long entities such as SWaT (~0.2 GB).